# Flowing-liquid activation and photon contact dose

Liquid moves through an irradiation region, travels to a storage pool, and mixes completely on arrival. We model **ideal plug flow in the pipes and perfect mixing in the pool**, without CFD. The liquid, total irradiated volume, total flow, travel time and initial pool inventory are replaceable inputs. A separate neutron-transport model supplies the reaction rates; its geometry need not represent an individual pipe.

This example uses constant flow and one rectangular neutron pulse. Equivalent parallel branches are represented by their total volume and flow. The pipes initially contain fresh liquid, pumping is already established at source startup, and there is no recirculation, pool outflow or chemical loss. The pool and transit section are outside the neutron field.

Activation and decay use OpenMC's full nuclide chain and CRAM solver. The neutron spectrum and material density are held fixed. Pumping may stop at shutdown or continue to flush the path; liquid still in the pipes is not added instantly to the pool. The reported photon material-slab dose is not a finite-pool transport calculation or a guaranteed conservative bound.

## Nuclear data and imports

Use matching neutron cross sections and an activation chain with decay-photon spectra for the liquid being modelled. Water requires O16(n,p)N16 and the photons from N16 decay. Other liquids require their corresponding activation pathways; the flow calculation does not select particular nuclides. OpenMC's `Chain.form_matrix`, `ReactionRates`, CRAM solver and material photon contact-dose method are used below.

The paths are workshop defaults. Existing `OPENMC_CROSS_SECTIONS` and `OPENMC_CHAIN_FILE` settings are respected. Change the paths for your installation. Calculation files are kept in a dedicated directory and are replaced when the transport calculation is rerun.

In [ ]:
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from scipy.sparse import bmat, csc_matrix, eye
import openmc
import openmc.deplete
from openmc.deplete.cram import CRAM48

openmc.config.setdefault('cross_sections', Path.home() / 'nuclear_data/cross_sections.xml')
openmc.config.setdefault('chain_file', Path.home() / 'nuclear_data/chain-endf-b8.0.xml')

output_directory = Path('flowing_liquid_results').resolve()
output_directory.mkdir(exist_ok=True)

## Define the liquid

Replace this block with any `openmc.Material` containing the desired composition and density. Here we use water at 1 g/cm3. No temperature or bound-atom thermal-scattering law is set. All activation products are assumed to remain in the liquid; chemical separation and gas release are not modelled.

In [ ]:
liquid = openmc.Material(name='water')
liquid.add_element('H', 2.0)
liquid.add_element('O', 1.0)
liquid.set_density('g/cm3', 1.0)

## Set the flow, pulse and pool inputs

`liquid.volume` and `mass_flow_rate_kg_s` are totals across the irradiated branches. We assume equal irradiated lengths and internal cross-sections, equal flow sharing, comparable neutron exposure, and a common transit time to the pool. Equal lengths alone are insufficient. Under these assumptions, pipe count cancels:

$$\tau = \frac{\rho V_{\mathrm{total}}}{\dot m_{\mathrm{total}}}.$$

The initial pool contains the same liquid composition. `transit_time_s` is prescribed, not calculated from pipe geometry. `pump_after_shutdown_s = None` continues pumping until the last irradiated liquid has reached the pool; set it to zero to stop pumping at source shutdown, or to a duration in seconds. Flushing adds fresh liquid and therefore affects pool dilution. Initially fresh liquid already in the transit section also reaches the pool.

A zero flow leaves the pipe inventory stationary: only the initial pool inventory contributes to pool dose. The method also handles residence times longer than the irradiation pulse. The numerical values below are illustrative, not restrictions on the model.

In [ ]:
liquid.volume = 10.0  # Total irradiated liquid volume across all branches [cm3]
liquid.depletable = True
mass_flow_rate_kg_s = 1.0  # Total flow through those branches
irradiation_time_s = 5.0
total_source_neutrons = 1.0e19
transit_time_s = 2.0  # Travel time from the irradiation region to the pool
initial_pool_mass_kg = 0.0
pump_after_shutdown_s = None  # None: flush the path; 0: stop at shutdown
cooling_times_s = np.array([0.0, 1.0, 5.0, 10.0, 30.0, 60.0, 300.0])

source_rate = total_source_neutrons / irradiation_time_s
volume_flow_rate = mass_flow_rate_kg_s * 1000.0 / liquid.get_mass_density()  # cm3/s
residence_time_s = liquid.volume / volume_flow_rate if volume_flow_rate else np.inf
initial_pool_volume = initial_pool_mass_kg * 1000.0 / liquid.get_mass_density()
if pump_after_shutdown_s is None:
    flush_time = residence_time_s + transit_time_s if volume_flow_rate else 0.0
    pump_stop_s = irradiation_time_s + flush_time
else:
    pump_stop_s = irradiation_time_s + pump_after_shutdown_s

if (liquid.volume <= 0 or liquid.get_mass_density() <= 0 or irradiation_time_s <= 0
        or mass_flow_rate_kg_s < 0 or transit_time_s < 0 or initial_pool_mass_kg < 0
        or pump_stop_s < irradiation_time_s):
    raise ValueError('Volumes, density and irradiation time must be positive; other inputs nonnegative.')
print(f'Source strength: {source_rate:.3e} neutrons/s')
print(f'Irradiation residence time: {residence_time_s:g} s')
print(f'Pump stops at {pump_stop_s:g} s after the source starts')

## Supply a neutron-transport model

The sphere below is only a compact example for calculating neutron exposure. Replace the geometry and source cells with an OpenMC model of the actual irradiation region, using `liquid` for the domain being tallied and its total occupied volume. The flow calculation uses the resulting reaction rates, not a sphere radius, pipe diameter or pipe count.

A single set of volume-averaged rates represents a uniform effective exposure along equivalent paths. Strongly different branch exposures or spatially varying irradiation histories require separate path or segment rates. Geometry still matters to the neutron field; it is not inferred from volume and flow alone.

In [ ]:
radius = (3.0 * liquid.volume / (4.0 * np.pi)) ** (1.0 / 3.0)
sphere = openmc.Sphere(r=radius)
source_distance = radius + 5.0  # Centre-to-source distance [cm]
boundary = openmc.Sphere(r=source_distance + 10.0, boundary_type='vacuum')
geometry = openmc.Geometry([
    openmc.Cell(fill=liquid, region=-sphere),
    openmc.Cell(region=+sphere & -boundary),
])

## Define the source and transport settings

Neutrons are emitted isotropically at 14 MeV in this example. Set the energy distribution directly in the source. Monte Carlo tallies use unit source strength; the physical neutron rate is applied once when constructing the activation matrix. Simulated histories control statistical precision, not the physical number of source neutrons.

In [ ]:
settings = openmc.Settings()
settings.run_mode = 'fixed source'
settings.batches = 50
settings.particles = 20_000
settings.seed = 1
settings.source = openmc.IndependentSource(
    space=openmc.stats.Point((0.0, 0.0, -source_distance)),
    angle=openmc.stats.Isotropic(),
    energy=openmc.stats.Discrete([14.0e6], [1.0]),
    strength=1.0,
)
model = openmc.Model(geometry, openmc.Materials([liquid]), settings)

## Run neutron transport once

Reduce the chain to nuclides reachable from the supplied liquid and obtain spectrum-weighted microscopic cross sections. Check nuclear-data coverage, chain-depth and Monte Carlo convergence for the intended exposure. Products without neutron cross sections can decay, but further neutron reactions on them are omitted.

The calculated spectrum is held fixed. The nuclide equations still include target depletion, reaction branching and daughter ingrowth; significant changes to the neutron field require refreshed transport. The one-domain example below can be replaced by reaction-rate data from another geometry without changing the flow equations.

In [ ]:
chain = openmc.deplete.Chain.from_xml(openmc.config['chain_file'])
chain = chain.reduce(liquid.get_nuclides(), level=5)
chain_file = output_directory / 'activation_chain.xml'
chain.export_to_xml(chain_file)

fluxes, micros = openmc.deplete.get_microxs_and_flux(
    model, [liquid], chain_file=chain,
    run_kwargs={'cwd': output_directory / 'transport', 'threads': 2},
)

## Construct the activation and decay equations

The microscopic cross sections and volume-integrated flux tally give reaction probabilities per atom per source neutron after dividing by the domain volume and converting barns to cm2. Multiplying by the physical source rate gives reactions per atom per second.

OpenMC builds an irradiation matrix, containing reactions and decay, and a decay-only matrix. These operate on all chain nuclides. A parcel with atom-density vector $n$ evolves as $n(t)=\exp(At)n(0)$ during irradiation and with the decay matrix after leaving the neutron field. We use the same CRAM solver as OpenMC depletion; no water-specific production formula is used.

In [ ]:
# Convert volume-integrated tallies per source neutron into reactions/atom/s.
rates = openmc.deplete.ReactionRates(['path'], micros[0].nuclides, chain.reactions)
for name in micros[0].nuclides:
    for reaction in chain.reactions:
        rate = (micros[0][name, reaction] * fluxes[0]).sum()
        rates.set('path', name, reaction, rate * 1.0e-24 * source_rate / liquid.volume)

irradiation_matrix = chain.form_matrix(rates[0])
decay_matrix = chain.form_matrix((rates * 0.0)[0])
nuclide_names = [nuclide.name for nuclide in chain.nuclides]
densities = liquid.get_nuclide_atom_densities()
fresh_density = np.array([densities.get(name, 0.0) * 1.0e24 for name in nuclide_names])

## Integrate continuous plug-flow arrivals

A finite pulse produces three arrival phases: increasing exposure during startup, a plateau, and decreasing exposure after shutdown. Their order depends on whether the pulse or residence time is shorter. Later arrivals are unirradiated fresh liquid. Transit delays every arrival from the irradiation region.

To integrate a continuously changing parcel composition without replacing fluid in artificial batches, we use block matrices. For example,

$$\frac{d}{dt}\begin{pmatrix}n\\z\end{pmatrix}=\begin{pmatrix}A&0\\I&D\end{pmatrix}\begin{pmatrix}n\\z\end{pmatrix}$$

simultaneously evolves the parcel under irradiation and accumulates its decaying contribution in $z$. Setting either diagonal block to zero gives the other integrals used below. CRAM evaluates these matrix exponentials. Output times do not set a fluid-replacement timestep.

In [ ]:
# Block matrices integrate parcel inventories over continuous arrival times.
# The first block evolves a parcel; the second accumulates its contribution.
nuclide_count = len(nuclide_names)
zeros = np.zeros(nuclide_count)
zero_matrix = csc_matrix((nuclide_count, nuclide_count))
identity = eye(nuclide_count, format='csc')
growing = bmat([[irradiation_matrix, None], [identity, decay_matrix]], format='csc')
constant = bmat([[zero_matrix, None], [identity, decay_matrix]], format='csc')
accumulating = bmat([[irradiation_matrix, None], [identity, zero_matrix]], format='csc')

arrival_phases = [
    (transit_time_s, transit_time_s + min(irradiation_time_s, residence_time_s), 'startup'),
    (transit_time_s + min(irradiation_time_s, residence_time_s),
     transit_time_s + max(irradiation_time_s, residence_time_s), 'plateau'),
    (transit_time_s + max(irradiation_time_s, residence_time_s),
     transit_time_s + irradiation_time_s + residence_time_s, 'shutdown'),
    (transit_time_s + irradiation_time_s + residence_time_s, np.inf, 'fresh'),
]
report_times_s = np.unique(np.r_[
    np.linspace(0.0, irradiation_time_s, 21), irradiation_time_s + cooling_times_s,
    pump_stop_s, [stop for _, stop, _ in arrival_phases],
])
report_times_s = report_times_s[np.isfinite(report_times_s)]

## Accumulate the pool inventory

For each observation time, integrate only liquid that has arrived before both that time and pump shutdown. Each contribution includes activation during its actual overlap with the neutron pulse and decay during transit and pool storage.

The pool volume grows by volumetric flow multiplied by pumping time. If the pump stops before the path is flushed, activity still in the pipes is excluded from the pool result. There is no instantaneous transfer of pipe contents at neutron shutdown.

In [ ]:
pool_inventories = []
pool_volumes = []
for time in report_times_s:
    arrival_end = min(time, pump_stop_s)
    # Initially the transit pipe contains fresh liquid, already present at t=0.
    initial_volume = initial_pool_volume + volume_flow_rate * min(arrival_end, transit_time_s)
    pool_atoms = CRAM48(decay_matrix, fresh_density * initial_volume, time)

    if volume_flow_rate > 0.0:
        for start, stop, phase in arrival_phases:
            stop = min(stop, arrival_end)
            if stop <= start:
                continue
            duration = stop - start
            if phase == 'startup':
                parcel = CRAM48(irradiation_matrix, fresh_density, start - transit_time_s)
                matrix = growing
                decay_time = time - stop + transit_time_s
            elif phase == 'plateau' and irradiation_time_s >= residence_time_s:
                parcel = CRAM48(irradiation_matrix, fresh_density, residence_time_s)
                matrix = constant
                decay_time = time - stop + transit_time_s
            elif phase == 'plateau':
                # A short pulse ends before the initially filled pipe is replaced.
                parcel = CRAM48(irradiation_matrix, fresh_density, irradiation_time_s)
                pool_atoms += volume_flow_rate * duration * CRAM48(
                    decay_matrix, parcel, time - irradiation_time_s)
                continue
            elif phase == 'shutdown':
                exposure = irradiation_time_s + residence_time_s + transit_time_s - stop
                parcel = CRAM48(irradiation_matrix, fresh_density, exposure)
                matrix = accumulating
                decay_time = time - irradiation_time_s
            else:
                parcel = fresh_density
                matrix = constant
                decay_time = time - stop + transit_time_s + residence_time_s

            # CRAM integrates each arrival interval, including radioactive decay.
            integrated = CRAM48(matrix, np.r_[parcel, zeros], duration)[nuclide_count:]
            pool_atoms += volume_flow_rate * CRAM48(
                decay_matrix, integrated, max(0.0, decay_time))

    pool_inventories.append(pool_atoms)
    pool_volumes.append(initial_pool_volume + volume_flow_rate * arrival_end)

## Convert pool inventories to concentrations

All liquid reaching the storage pool is assumed to mix immediately. Divide its nuclide atom counts by the actual accumulated pool volume before calculating material contact dose. This includes any initial pool inventory and fresh liquid added during flushing.

An initially empty pool has no material or dose until liquid arrives. With zero flow it stays empty, even though stationary liquid in the irradiation region may become activated.

In [ ]:
pool_materials = []
for inventory, volume in zip(pool_inventories, pool_volumes):
    if volume == 0.0:
        pool_materials.append(None)  # No liquid has reached an initially empty pool.
        continue
    material = openmc.Material(name='mixed storage pool')
    material.volume = volume
    material.set_density('sum')
    for name, atoms in zip(nuclide_names, inventory):
        if atoms > 0.0:
            material.add_nuclide(name, atoms / volume * 1.0e-24)  # atoms/b-cm
    pool_materials.append(material)

## Calculate mixed-pool activity and photon contact dose

The material method returns **absorbed-air photon dose in Gy/h**, using a slab approximation and a build-up factor of 2. It is not a personal dose in Sv/h or a photon-transport calculation around the actual pool. Direct beta, bremsstrahlung and delayed-neutron doses are excluded, and Monte Carlo and nuclear-data uncertainties are not propagated.

Pool dose can continue rising after neutron shutdown while activated liquid is still arriving. Decay is included both before and after arrival. The table distinguishes time from source startup and time from shutdown.

In [ ]:
dose_by_nuclide = [
    material.get_photon_contact_dose_rate(
        dose_quantity='absorbed-air', build_up=2.0, by_nuclide=True)
    if material is not None else {}
    for material in pool_materials
]
summary = pd.DataFrame({
    'Time from source start [s]': report_times_s,
    'Time from shutdown [s]': report_times_s - irradiation_time_s,
    'Pool volume [cm3]': pool_volumes,
    'Pool activity [Bq]': [material.get_activity(units='Bq') if material is not None else 0.0
                           for material in pool_materials],
    'Photon contact dose [Gy/h]': [sum(values.values()) for values in dose_by_nuclide],
})

## Inspect the pool results

The pool volume includes initially stored liquid and all flow delivered before pump shutdown. Negative values in the shutdown-time column denote times during irradiation. These results describe only the storage pool, not dose from irradiated liquid retained in the pipes.

In [ ]:
summary

## Plot the pool history and its contributors

The source-stop and pump-stop markers show whether the pool is still filling after irradiation. The contributor table uses the largest **sampled** pool dose, not an assumed maximum at shutdown. Refine the observation times when locating a peak. N16 is important for short-cooled water, but the same calculation follows all nuclides in another liquid's activation chain.

In [ ]:
figure, axes = plt.subplots(1, 2, figsize=(11, 4), layout='constrained')
axes[0].plot(summary['Time from source start [s]'], summary['Pool activity [Bq]'])
axes[0].set(xlabel='Time from source start [s]', ylabel='Pool activity [Bq]')
axes[1].plot(summary['Time from source start [s]'], summary['Photon contact dose [Gy/h]'])
axes[1].set(xlabel='Time from source start [s]', ylabel='Photon contact dose [Gy/h]')
axes[1].set_yscale('symlog', linthresh=1.0e-12)
for axis in axes:
    axis.axvline(irradiation_time_s, color='grey', linestyle='--', label='Source stops')
    axis.axvline(pump_stop_s, color='grey', linestyle=':', label='Pump stops')
    axis.legend()
plt.show()

In [ ]:
peak_index = summary['Photon contact dose [Gy/h]'].idxmax()
print('Largest sampled dose at', report_times_s[peak_index], 's from source start')
pd.Series(dose_by_nuclide[peak_index], dtype=float, name='Photon contact dose [Gy/h]').sort_values(
    ascending=False).head(10)

## Explore other conditions

Change the liquid material, total irradiated volume, total branch flow, transit delay, initial pool mass or pumping duration. The calculation handles irradiation shorter or longer than residence time without assuming a particular isotope half-life. Rerun transport when changing the liquid or neutron geometry.

The one-path model assumes equivalent parallel branches and a uniform effective exposure. Unequal branches, axial exposure variations, time-dependent flow, repeated pulses, recirculation or pool outflow need corresponding extensions to the path history or inventory balance. No CFD is required for prescribed histories, but those histories cannot be inferred from total volume and flow alone. The photon material-slab result is a screening estimate, not a guaranteed conservative finite-pool dose.